# Check metadata and register subjects in entitycore

Requirements:
- [entitysdk](https://github.com/openbraininstitute/entitysdk) v0.7.2 or higher!!
- rich
- openpyxl

In [1]:
import numpy as np
import pandas as pd
import os
import shutil
import time

from conntility import ConnectivityMatrix
from datetime import datetime, timedelta
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from pathlib import Path
from rich import print as rprint

In [2]:
token = get_token(environment="staging")
project_context = ProjectContext.from_vlab_url("https://staging.openbraininstitute.org/app/virtual-lab/lab/1f91f30e-1489-4e2a-8eb7-1217257c8e19/project/7a411785-6895-4839-aaa2-d9f76e09875a/home")
client = Client(environment="staging", project_context=project_context, token_manager=token)

# token = get_token(environment="production")
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment="production", project_context=project_context, token_manager=token)

In [3]:
if client.project_context.environment == "staging":
    ID_key = "entityCoreIDStaging"
else:
    ID_key = "entityCoreID"

In [4]:
DAYS_PER_YEAR = 365.2425  # Mean length of the Gregorian calendar year

## List of subjects to be registered

In [5]:
# Root path with all the hand curated metadata in OneDrive
input_root = "/Users/pokorny/OneDrive - Open Brain Institute/Circuits hardcoded"


In [6]:
circuits_file = os.path.join(input_root, "Circuit list.xlsx")
sheet_name = "Subjects"  # Sheet with subjects
subjects_df = pd.read_excel(circuits_file, sheet_name=sheet_name)
subjects_df

,entityCoreID,entityCoreIDStaging,species,subjectName,subjectDescription,subjectAge,subjectAgeUnit,subjectAgePeriod,subjectSex
0,f0b0e4d3-2cc9-4266-b7c8-c0e284666315,e5ecb660-504f-4840-b674-f31f0eada439,Rattus norvegicus,Average rat P14,Abstract representation of a P14 rat model bui...,14.0,days,postnatal,unknown
1,38bf9782-9c98-4490-802d-61e9a8584b35,445edb11-40ee-431f-b9d2-e9e96deab8f9,Rattus norvegicus,Average rat,Abstract representation of a rat model built f...,NaN,NaN,NaN,unknown
2,4fa6d019-6283-413e-a136-4b506314e555,8851f1e7-0fa8-4d31-84c5-e76dc3383fb7,Homo sapiens,H01 W45,"Subject from Harvad's data set H01, correspond...",45.0,years,postnatal,female
3,760bcba1-c49e-469d-9572-f1497430cb57,d78f4975-bd36-446b-b3e6-a8247588018a,Mus musculus,IARPA MICrONS mouse,This entity represents the the mouse sacrifice...,87.0,days,postnatal,male
4,NaN,NaN,Mus musculus,Average mouse,Abstract representation of a mouse model built...,NaN,NaN,NaN,unknown


#### Find or create subjects

In [11]:
make_public = True
check_only = False

In [12]:
entity_df = subjects_df[[]].copy()
for idx, subject_row in subjects_df.iterrows():
    if not str(subject_row[ID_key]) == "nan":
        continue  # Skip if ID already in table

    # Find species
    species = client.search_entity(
        entity_type=models.Species, query={"name__ilike": subject_row["species"]}
    ).first()
    
    assert species is not None, f"ERROR: Species '{subject_row['species']}' not found!"

    # Find subject, if already existing
    subjects = client.search_entity(
        entity_type=models.Subject, query={"name": subject_row["subjectName"]}
    ).all()

    if len(subjects) > 0:
        # Check consistency and use existing subject
        assert len(subjects) == 1, f"ERROR: Multiple subject entities with name '{subject_row['subjectName']}' found!"
        subject = subjects[0]
        assert subject.description == subject_row["subjectDescription"], "ERROR: Subject description mismatch!"
        assert subject.sex == subject_row["subjectSex"], "ERROR: Subject description mismatch!"
        assert subject.species.id == species.id, "ERROR: Subject species mismatch!"
        if np.isfinite(subject_row["subjectAge"]):
            age = subject_row["subjectAge"]
            age_unit = subject_row["subjectAgeUnit"]
            age_value = timedelta(**{age_unit: age})
            assert subject.age_value == age_value
        else:
            assert subject.age_value is None
            assert subject.age_min is None
            assert subject.age_max is None
        if isinstance(subject_row["subjectAgePeriod"], str):
            assert subject.age_period == subject_row["subjectAgePeriod"]
        else:
            assert subject.age_period is None
        print("Using existing subject:", end=" ")
    else:
        # Register new subject
        age_dict = {}
        if np.isfinite(subject_row["subjectAge"]):
            age = subject_row["subjectAge"]
            age_unit = subject_row["subjectAgeUnit"]
            if age_unit == "years":  # Not supported by timedelta, so convert to days
                age = int(np.round(age * DAYS_PER_YEAR))
                age_unit = "days"
            age_dict["age_value"] = timedelta(**{age_unit: age})
        if isinstance(subject_row["subjectAgePeriod"], str):
            age_dict["age_period"] = subject_row["subjectAgePeriod"]
        subject = models.Subject(
            name=subject_row["subjectName"],
            description=subject_row["subjectDescription"],
            authorized_public=make_public,
            sex=subject_row["subjectSex"],
            species=species,
            **age_dict
        )
        if check_only:
            print("CHECK ONLY:", end=" ")
        else:
            subject = client.register_entity(subject)
            print("Registered new subject:", end=" ")
    print(f"{subject.name}, ID {subject.id}")
    entity_df.loc[idx, "name"] = subject.name
    entity_df.loc[idx, ID_key] = subject.id

Registered new subject: Average mouse, ID f0b42847-2a27-4726-9fde-7423fcde5fef


In [13]:
# List of registered subjects
entity_df

,name,entityCoreIDStaging
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,Average mouse,f0b42847-2a27-4726-9fde-7423fcde5fef


In [14]:
# Export to .csv file
csv_file = f"registered_subjects_{project_context.environment.value}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
entity_df.to_csv(csv_file)